In [ ]:
from pathlib import Path

DATA_DIR = Path("Data")
if not DATA_DIR.exists():
    raise FileNotFoundError("Expected Data/ in the current working directory. Run this notebook from the project root.")

print(f"Working directory: {Path.cwd()}")


In [ ]:
# Manual counts from the search outputs; used to allocate videos proportionally.
videos_per_term = {
    "Bach Flowers": 98,
    "Reiki": 98,
    "Acupuncture": 75,
    #"Magnetotherapy": 99,
    "Energetic Therapy": 60,
    "Naturopathy": 31,
    "Cognitive Behavioral Therapy": 83,
    "Cancer treatment": 67,
    "Cardiac rehabilitation": 38,
    "Sports physiotherapy": 20,
    "Endoscopic procedure": 9
}

# Search terms are grouped by audience before computing sample sizes.
pseudoscientific_terms = [
    "Bach Flowers",
    "Reiki",
    "Acupuncture",
    #"Magnetotherapy",
    "Energetic Therapy",
    "Naturopathy"
]

scientific_terms = [
    "Cognitive Behavioral Therapy",
    "Cancer treatment",
    "Cardiac rehabilitation",
    "Sports physiotherapy",
    "Endoscopic procedure"
]


def select_videos(terms, target=15):
    """
    Select a fixed number of videos from the given terms.

    Each term receives at least one video. The remaining videos are
    distributed proportionally based on how many videos each term has.
    """

    # Get the available video count for each selected term.
    video_counts = {}

    for term in terms:
        video_counts[term] = videos_per_term[term]

    # Start by assigning one video to every term.
    videos_selected = {}

    for term in terms:
        videos_selected[term] = 1

    # Calculate how many videos still need to be assigned.
    videos_remaining = target - len(terms)

    # Calculate the total number of available videos in this group.
    total_available_videos = sum(video_counts.values())

    # Distribute the remaining videos proportionally.
    for term, available_count in video_counts.items():
        proportion = available_count / total_available_videos
        additional_videos = round(proportion * videos_remaining)

        videos_selected[term] += additional_videos

    # Rounding may result in fewer videos than the target.
    # Add missing videos to the term with the largest available count.
    while sum(videos_selected.values()) < target:
        term_with_most_videos = max(
            video_counts,
            key=video_counts.get
        )

        videos_selected[term_with_most_videos] += 1

    # Rounding may result in more videos than the target.
    # Remove videos from smaller categories first, without going below one.
    while sum(videos_selected.values()) > target:
        terms_from_smallest_to_largest = sorted(
            video_counts,
            key=video_counts.get
        )

        for term in terms_from_smallest_to_largest:
            if videos_selected[term] > 1:
                videos_selected[term] -= 1
                break

    return videos_selected


# Compute the planned number of sampled videos per term.
pseudoscientific_selection = select_videos(pseudoscientific_terms)
scientific_selection = select_videos(scientific_terms)

print("Pseudoscientific:", pseudoscientific_selection)
print("Total:", sum(pseudoscientific_selection.values()))

print("Scientific:", scientific_selection)
print("Total:", sum(scientific_selection.values()))

In [ ]:
from pathlib import Path

base = Path("Data") / "Videos"

# Quick inventory of the available search-result CSV files.

for csv_path in base.rglob("*.csv"):
    print(csv_path)

**Mapping**

In [ ]:
from pathlib import Path

# Generate the sample sizes used by the video sampling loop.
pseudoscientific_selection = select_videos(
    pseudoscientific_terms,
    target=15
)

scientific_selection = select_videos(
    scientific_terms,
    target=15
)

# Combine both results into one dictionary
sample_sizes = {
    **pseudoscientific_selection,
    **scientific_selection
}

# Map the English analysis labels to the original CSV filenames.
term_to_csv = {
    "Magnetotherapy": "terms_pseudo/Busqueda_Magnetoterapia.csv",
    "Bach Flowers": "terms_pseudo/Busqueda_Flores de Bach.csv",
    "Reiki": "terms_pseudo/Busqueda_Reiki.csv",
    "Acupuncture": "terms_pseudo/Busqueda_Acupuntura.csv",
    "Energetic Therapy": "terms_pseudo/Busqueda_Terapia energética.csv",
    "Naturopathy": "terms_pseudo/Busqueda_Naturopatía.csv",
    "Cognitive Behavioral Therapy": (
        "terms_cient/Busqueda_Terapia Cognitivo Conductual.csv"
    ),
    "Cancer treatment": "terms_cient/Busqueda_Tratamiento cáncer.csv",
    "Cardiac rehabilitation": (
        "terms_cient/Busqueda_Rehabilitación cardíaca.csv"
    ),
    "Sports physiotherapy": (
        "terms_cient/Busqueda_Fisioterapia deportiva.csv"
    ),
    "Endoscopic procedure": (
        "terms_cient/Busqueda_Procedimiento endóscopico.csv"
    ),
}

base_dir = Path("Data") / "Videos"

# Print the exact source file and requested sample size for each term.
for term, sample_size in sample_sizes.items():
    csv_path = base_dir / term_to_csv[term]

    print(
        f"{term}: {csv_path} -> {sample_size} rows"
    )

**Getting videos**

In [ ]:
from pathlib import Path
import csv
import random
import pandas as pd


random.seed(27)

base_dir = Path("Data") / "Videos"

rows_with_label = []
rows_without_label = []

# Build two aligned exports: one with the model label and one blank for annotators.
for term, sample_size in sample_sizes.items():

    csv_path = base_dir / term_to_csv[term]

    with csv_path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        video_id_column = reader.fieldnames[0]
        rows = list(reader)

    # Eligibility mirrors the study inclusion rules for video-level sampling.
    eligible_rows = [
        row
        for row in rows
        if (
            row["duration_seconds"]
            and row["comment_count"]
            and int(row["duration_seconds"]) <= 300
            and int(row["comment_count"]) > 0
            and row["detected_language"] in ("en", "es")
        )
    ]

    if len(eligible_rows) < sample_size:
        raise ValueError(
            f"{term}: only {len(eligible_rows)} eligible videos, "
            f"but {sample_size} were requested."
        )

    # Sample without replacement within the current search term.
    sampled_rows = random.sample(eligible_rows, sample_size)

    print(
        f"{term}: {len(eligible_rows)} eligible videos "
        f"(sampling {sample_size})"
    )

    for row in sampled_rows:

        video_id = row[video_id_column]
        video_url = f"https://www.youtube.com/watch?v={video_id}"

        # Internal version keeps the original classification for later checks.
        rows_with_label.append({
            "link": video_url,
            "video_title": row["video_title"],
            "video_description": row["video_description"],
            "etiqueta": row["Content_Classification"],
        })

        # Annotation version has the same video metadata but hides the label.
        rows_without_label.append({
            "link": video_url,
            "video_title": row["video_title"],
            "video_description": row["video_description"],
            "etiqueta": "",
        })


# Randomize the final order while keeping both Excel files row-aligned.
combined_rows = list(zip(rows_with_label, rows_without_label))
random.shuffle(combined_rows)

rows_with_label, rows_without_label = map(
    list,
    zip(*combined_rows),
)


df_with_label = pd.DataFrame(rows_with_label)
df_without_label = pd.DataFrame(rows_without_label)

# Export readable Excel files with wrapped descriptions and clickable links.
for output_file, dataframe in [
    ("sampled_videos_with_etiqueta.xlsx", df_with_label),
    ("sampled_videos_without_etiqueta.xlsx", df_without_label),
]:
    with pd.ExcelWriter(
        base_dir / output_file,
        engine="xlsxwriter",
        engine_kwargs={
            "options": {
                "strings_to_urls": False
            }
        },
    ) as writer:

        dataframe.to_excel(
            writer,
            sheet_name="Videos",
            index=False,
        )

        workbook = writer.book
        worksheet = writer.sheets["Videos"]

        wrap_format = workbook.add_format({
            "text_wrap": True,
            "valign": "top",
        })

        link_format = workbook.add_format({
            "font_color": "blue",
            "underline": True,
            "valign": "top",
        })

        worksheet.set_column("A:A", 45)
        worksheet.set_column("B:B", 50)
        worksheet.set_column("C:C", 100, wrap_format)
        worksheet.set_column("D:D", 30)

        # Add hyperlinks only to the video link column
        for row_number, video_url in enumerate(
            dataframe["link"],
            start=1,
        ):
            worksheet.write_url(
                row_number,
                0,
                video_url,
                link_format,
                string=video_url,
            )

        for row_number in range(1, len(dataframe) + 1):
            worksheet.set_row(row_number, 60)

print("\nExcel files created.")
print(f"Total videos: {len(rows_with_label)}")

**Sampling comments from scientific audience**

In [ ]:
import pandas as pd

# Load the evaluated comment datasets used for comment-level sampling.
pseudoscientific = pd.read_csv("Data//Comments//Evaluated_pseudo_multilingual.csv")
scientific = pd.read_csv("Data//Comments//Evaluated_scientific_multilingual.csv")

In [2]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


# Keep English and Spanish separate; all other languages form one stratum.
def prepare_language_column(df):
    df = df.copy()

    df["language"] = df["language"].where(
        df["language"].isin(["en", "es"]),
        "other"
    )

    return df


def create_stratified_sample(
    df,
    sample_size,
    columns_to_preserve,
    random_state=42
):

    # Convert the selected categorical columns into multilabel strata.
    stratification_matrix = pd.get_dummies(
        df[columns_to_preserve]
        .fillna("MISSING")
        .astype(str),
        dtype=int
    )

    # Draw one stratified split so language, emotion, and label proportions are preserved.
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        train_size=sample_size,
        test_size=len(df) - sample_size,
        random_state=random_state
    )

    sample_indices, remaining_indices = next(
        splitter.split(df, stratification_matrix)
    )

    sample_df = df.iloc[sample_indices].copy()
    remaining_df = df.iloc[remaining_indices].copy()

    return sample_df, remaining_df


# Load datasets
pseudoscientific = pd.read_csv(
    "Data/Comments/Evaluated_pseudo_multilingual.csv"
)

scientific = pd.read_csv(
    "Data/Comments/Evaluated_scientific_multilingual.csv"
)


# Group languages other than English and Spanish as "other"
pseudoscientific = prepare_language_column(pseudoscientific)
scientific = prepare_language_column(scientific)


# Sampling configuration: 50 comments per audience, preserving key annotation variables.
sample_size = 50
columns_to_preserve = ["language", "emotion", "etiqueta"]


# Create samples
scientific_sample, scientific_remaining = create_stratified_sample(
    df=scientific,
    sample_size=sample_size,
    columns_to_preserve=columns_to_preserve,
    random_state=42
)

pseudoscientific_sample, pseudoscientific_remaining = (
    create_stratified_sample(
        df=pseudoscientific,
        sample_size=sample_size,
        columns_to_preserve=columns_to_preserve,
        random_state=42
    )
)


# Add one extra scientific comment from "medical treatment" (after executing the previous code, only 49 comments were present
# in the scientific sample, so we add an extra one of an important category for this sample)
extra_medical_treatment = (
    scientific_remaining[
        scientific_remaining["etiqueta"]
        .str.strip()
        .str.lower()
        .eq("medical treatment")
    ]
    .sample(
        n=1,
        random_state=42
    )
)

scientific_sample = pd.concat(
    [
        scientific_sample,
        extra_medical_treatment
    ],
    ignore_index=True
)

scientific_remaining = scientific_remaining.drop(
    index=extra_medical_treatment.index
)


# Store the audience source before combining both samples.
scientific_sample["source"] = "scientific"
pseudoscientific_sample["source"] = "pseudoscientific"


# Combine both samples
combined_sample = pd.concat(
    [
        scientific_sample,
        pseudoscientific_sample
    ],
    ignore_index=True
)

# Combine the complete datasets to evaluate category coverage
full_dataset = pd.concat(
    [
        scientific.assign(source="scientific"),
        pseudoscientific.assign(source="pseudoscientific")
    ],
    ignore_index=True
)

# Categories present in the complete corpus and in the validation sample
full_categories = set(full_dataset["etiqueta"].dropna().unique())
sample_categories = set(combined_sample["etiqueta"].dropna().unique())

# Categories not represented in the validation sample
missing_categories = full_categories - sample_categories

print("Categories not represented in the validation sample:")
print(missing_categories)

# Frequency of those categories in the complete corpus
missing_category_counts = (
    full_dataset[
        full_dataset["etiqueta"].isin(missing_categories)
    ]["etiqueta"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="n_comments")
)

missing_category_counts["percent_of_full_corpus"] = (
    missing_category_counts["n_comments"]
    / len(full_dataset)
    * 100
)

display(missing_category_counts)

# Combined contribution of all categories missing from the sample
n_missing = missing_category_counts["n_comments"].sum()
pct_missing = n_missing / len(full_dataset) * 100

print(f"Total comments in complete corpus: {len(full_dataset):,}")
print(f"Comments belonging to unrepresented categories: {n_missing:,}")
print(f"Percentage of complete corpus: {pct_missing:.2f}%")


# General description of the combined sample
display(
    combined_sample[
        ["source", "language", "emotion", "etiqueta"]
    ].describe(include="all")
)


# Inspect whether the stratified sample retained comparable category distributions.
for column in columns_to_preserve:
    description = pd.concat(
        [
            pd.crosstab(
                combined_sample[column].fillna("MISSING"),
                combined_sample["source"]
            ),
            pd.crosstab(
                combined_sample[column].fillna("MISSING"),
                combined_sample["source"],
                normalize="columns"
            ).add_suffix("_proportion")
        ],
        axis=1
    )

    print(f"\nDescription for {column}")
    display(description.round(4))


# Description of all category combinations
combination_description = (
    combined_sample
    .fillna({
        column: "MISSING"
        for column in columns_to_preserve
    })
    .groupby(
        ["language", "emotion", "etiqueta", "source"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(combination_description)

Categories not represented in the validation sample:
{'Pseudoscientific treatment', 'Anecdote', 'Plans', 'Unclassifiable', 'Opinion Request', 'First impression', 'Emotional Support', 'Event anticipation', 'Spam', 'Opinion', 'Information Exchange', 'Joke', 'Tribute'}


,category,n_comments,percent_of_full_corpus
0,Joke,478,0.912022
1,Pseudoscientific treatment,476,0.908206
2,Anecdote,402,0.767015
3,Tribute,368,0.702143
4,Plans,217,0.414035
5,Unclassifiable,216,0.412127
6,Opinion,181,0.345347
7,Information Exchange,113,0.215604
8,Opinion Request,79,0.150732
9,Emotional Support,77,0.146916


Total comments in complete corpus: 52,411
Comments belonging to unrepresented categories: 2,760
Percentage of complete corpus: 5.27%


,source,language,emotion,etiqueta
count,100,100,100,100
unique,2,3,3,12
top,scientific,en,Positivo,Expression of personal feelings
freq,50,38,54,40



Description for language


...


Description for emotion


source,pseudoscientific,scientific,pseudoscientific_proportion,scientific_proportion
emotion,,,,
Negativo,12,23,0.24,0.46
Neutral,5,6,0.10,0.12
Positivo,33,21,0.66,0.42



Description for etiqueta


source,pseudoscientific,scientific,pseudoscientific_proportion,scientific_proportion
etiqueta,,,,
Advice Give,0,1,0.00,0.02
Advice request,1,2,0.02,0.04
Comparison,10,14,0.20,0.28
Compliment,6,4,0.12,0.08
Criticism,0,1,0.00,0.02
Desires,3,1,0.06,0.02
Expression of personal feelings,21,19,0.42,0.38
Greetings,1,1,0.02,0.02
Insult,2,1,0.04,0.02


source,language,emotion,etiqueta,pseudoscientific,scientific
0,en,Negativo,Advice request,0,1
1,en,Negativo,Comparison,5,2
2,en,Negativo,Compliment,1,0
3,en,Negativo,Expression of personal feelings,3,3
4,en,Negativo,Speculation,1,0
5,en,Neutral,Comparison,1,1
6,en,Neutral,Compliment,1,0
7,en,Neutral,Expression of personal feelings,2,1
8,en,Positivo,Comparison,1,1
9,en,Positivo,Compliment,0,1


In [ ]:
# Compact CSV used for quick manual inspection outside the notebook.
combined_sample
combined_sample[["comment", "emotion", "etiqueta"]].to_csv(
    "combined_sample_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Keep only the fields annotators need at this stage.
annotation_dataset = combined_sample[
    ["source", "language", "comment"]
].copy()

annotation_dataset = annotation_dataset.reset_index(drop=True)

# Add a stable row identifier for tracking annotations across files.
annotation_dataset.insert(
    0,
    "comment_id",
    range(1, len(annotation_dataset) + 1)
)

annotation_dataset = annotation_dataset.rename(
    columns={
        "language": "source_language",
        "comment": "original_comment"
    }
)

annotation_dataset["comment_es"] = ""
annotation_dataset["comment_en"] = ""

# The original Spanish text is already the Spanish version
spanish_mask = annotation_dataset["source_language"].eq("es")

annotation_dataset.loc[
    spanish_mask,
    "comment_es"
] = annotation_dataset.loc[
    spanish_mask,
    "original_comment"
]

# The original English text is already the English version
english_mask = annotation_dataset["source_language"].eq("en")

annotation_dataset.loc[
    english_mask,
    "comment_en"
] = annotation_dataset.loc[
    english_mask,
    "original_comment"
]

# Empty columns are placeholders to be filled during annotation.
annotation_dataset["annotation"] = ""
annotation_dataset["annotator"] = ""
annotation_dataset["notes"] = ""

annotation_dataset.head()

In [ ]:
import requests
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown

# Helper used only to display model responses more readably if needed.
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

import os
GOOGLE_API_KEY=os.getenv('')
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-3.1-flash-lite")
genai.configure(api_key='')

In [ ]:
PROMPT_TEMPLATE = """
You are a professional translator.

Translate the following text into {target_language}.

Preserve the original meaning and tone. Do not summarize, explain, or add information.

Return only the translated text.

Text:
{comment}
"""

In [ ]:
import time
# Translate one comment at a time using the prompt template above.
def translate(comment, target_language):
    prompt = PROMPT_TEMPLATE.format(
        target_language=target_language,
        comment=comment
    )
    time.sleep(5)
    response = model.generate_content(prompt)

    return response.text.strip()


translated_rows = []

# Build bilingual columns; existing English/Spanish comments are not translated again.
for _, row in combined_sample.iterrows():
    comment = row["comment"]
    language = row["language"]

    if language == "es":
        comment_es = comment
        comment_en = translate(comment, "English")

    elif language == "en":
        comment_es = translate(comment, "Spanish")
        comment_en = comment

    else:
        comment_es = translate(comment, "Spanish")
        comment_en = translate(comment, "English")

    translated_rows.append({
        "original_comment": comment,
        "comment_es": comment_es,
        "comment_en": comment_en
    })


# Join translations back to the sampled comments for annotation export.
translations = pd.DataFrame(translated_rows)

annotation_dataset = pd.concat(
    [
        combined_sample.reset_index(drop=True),
        translations
    ],
    axis=1
)

In [ ]:
import pandas as pd
from openpyxl.styles import Alignment


# Use labels present in the sample as annotation columns, plus an open Other option.
labels = sorted(combined_sample["etiqueta"].dropna().unique())
labels.append("Other")


# Base columns
comment_columns = [
    "original_comment",
    "comment_es",
    "comment_en",
]


# Internal file keeps the original label for checking against annotations.
comments_with_etiqueta = annotation_dataset[
    comment_columns + ["etiqueta"]
].copy()


# Annotator-facing file hides the original label.
comments_without_etiqueta = annotation_dataset[
    comment_columns
].copy()

comments_without_etiqueta["etiqueta"] = ""


# Strip whitespace before exporting and add empty annotation fields.
for dataframe in [
    comments_with_etiqueta,
    comments_without_etiqueta,
]:
    dataframe[comment_columns] = dataframe[comment_columns].apply(
        lambda column: column.astype(str).str.strip()
    )

    dataframe["emotion"] = ""

    for label in labels:
        dataframe[label] = ""


# Export both workbooks with wrapping and frozen headers for manual annotation.
for filename, dataframe in [
    ("comments_with_etiqueta.xlsx", comments_with_etiqueta),
    ("comments_for_annotation.xlsx", comments_without_etiqueta),
]:
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:

        dataframe.to_excel(
            writer,
            index=False,
            sheet_name="Annotations",
        )

        worksheet = writer.sheets["Annotations"]

        for row in worksheet.iter_rows():
            for cell in row:
                cell.alignment = Alignment(
                    wrap_text=True,
                    vertical="top",
                )

        worksheet.column_dimensions["A"].width = 50
        worksheet.column_dimensions["B"].width = 50
        worksheet.column_dimensions["C"].width = 50
        worksheet.column_dimensions["D"].width = 20

        for column in worksheet.iter_cols(
            min_col=5,
            max_col=worksheet.max_column,
        ):
            worksheet.column_dimensions[
                column[0].column_letter
            ].width = 15

        worksheet.freeze_panes = "A2"


print("Excel files created.")